In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e11/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e11/train.csv
/kaggle/input/competitions/playground-series-s6e11/test.csv


In [ ]:
train_data = pd.read_csv('/Users/aleksandr/Документы/Predicting Credit Card Fraud/train.csv')
test_data = pd.read_csv('/Users/aleksandr/Документы/Predicting Credit Card Fraud/test.csv')

x_train = train_data.drop(columns=["is_fraud", "id"], errors="ignore").select_dtypes(include="number")
y_train = train_data["is_fraud"]

x_train, x_test, y_train, y_test = train_test_split(
        x_train, y_train, test_size=0.2, stratify=y_train
    )

In [3]:
model = RandomForestClassifier(
    bootstrap=True,
    n_jobs=-1
)
model.fit(x_train, y_train)

train_accuracy = accuracy_score(y_train, model.predict(x_train))
test_accuracy = accuracy_score(y_test, model.predict(x_test))
print(f"Точность обучения : {100* train_accuracy:.1f}%" )
print(f"Точность валидации : {100* test_accuracy:.1f}%" )

Точность обучения : 100.0%
Точность валидации : 98.4%


In [4]:
x_submission = test_data[x_train.columns]

fraud_probabilities = model.predict_proba(x_submission)[:, 1]

submission = pd.DataFrame({
    "id": test_data["id"],
    "is_fraud": fraud_probabilities
})

submission = submission[submission["is_fraud"] != 0]

submission.to_csv("submission.csv", index=False)

with open("submission.csv", "a", encoding="utf-8") as f:
    f.write("etc.\n")

print("File submission.csv successfully created")
print(submission.head())
print(f"Количество строк: {len(submission)}")

File submission.csv successfully created
        id  is_fraud
3   699963      0.01
4   699964      0.02
8   699968      0.02
9   699969      0.01
10  699970      0.01
Количество строк: 134202


In [5]:
preds = model.predict_proba(x_submission)[:, 1]

submission = pd.DataFrame({
    "id": test_data["id"].values,
    "is_fraud": preds
})

submission["is_fraud"] = submission["is_fraud"].fillna(0)

submission.to_csv("submission.csv", index=False)
